# Day 05 — ALL IN ONE — DINOv3 ViT-S/16

**Chỉ mở notebook này** để chạy: cache Day 04 → export GT native → khảo sát và khóa tiny/boundary → preflight → train R0/R1/R2 → inference → evaluation.

Dùng cấu hình Day 04: frozen `dinov3_vits16`, blocks 4/8/12, adapter r32/d384, projection 64, seed 42, 150 epochs, batch 64, AdamW lr 0.001, AMP bfloat16, checkpoint theo minimum val loss. Phase 1 chỉ Fabric.

1. Upload **`reports/day05_code_overlay.zip` mới đi kèm notebook này** lên Drive. ZIP chứa code; notebook tự kiểm tra SHA và giải nén.
2. Chọn runtime GPU. Sửa các đường dẫn trong **Cấu hình đường dẫn** và các ngưỡng trong **Cấu hình khảo sát GT**.
3. Chạy theo thứ tự từ trên xuống. Tại **Khóa protocol**, đọc báo cáo GT, thống nhất quy tắc, điền căn cứ và đặt `EXPORT_LOCKED_PROTOCOLS=True` khi thực sự đã khóa trước candidate results. Chạy cell khóa và cell nối protocol, rồi tiếp tục training/inference/evaluation.

Không cần chạy từng `.py` hoặc mở notebook khảo sát riêng. Cache Day 04 đã đủ thì không retrain Day 04. Notebook không tải dataset hoặc DINO weights. Code đã được kiểm tra trên fixture; kết quả nghiên cứu chỉ có sau khi chạy với dữ liệu/weights/GPU thật.

## 1. Setup Colab và nạp code đã kiểm tra SHA

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, subprocess, json, hashlib, zipfile, shutil

drive.mount('/content/drive', force_remount=False)
PROJECT_DRIVE = Path('/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9')
PROJECT_ROOT = Path('/content/msila')
OVERLAY = PROJECT_DRIVE / 'msila_day05/code/day05_code_overlay.zip'
OVERLAY_SHA256 = 'a2963dc169a0a9b0882c8f3f6078c513164f61841f2382e07e1fc2944900f194'
if not OVERLAY.is_file():
    raise FileNotFoundError(f'Upload reports/day05_code_overlay.zip vào {OVERLAY}')
if hashlib.sha256(OVERLAY.read_bytes()).hexdigest() != OVERLAY_SHA256:
    raise RuntimeError('Overlay SHA mismatch: dùng ZIP đi kèm notebook này')
if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git','clone','https://github.com/haduckien-raccoon/msila.git',str(PROJECT_ROOT)], check=True)
with zipfile.ZipFile(OVERLAY) as z:
    for entry in z.infolist():
        p = Path(entry.filename)
        if p.is_absolute() or '..' in p.parts:
            raise RuntimeError('Unsafe overlay entry')
    z.extractall(PROJECT_ROOT)
print('Code overlay verified:', OVERLAY_SHA256)

In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('BLOCKED_MISSING_GPU: chọn runtime GPU')
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','pyyaml','scipy','pillow','matplotlib'],check=True)

## 2. Cấu hình đường dẫn

Sửa RAW, DINO_REPO, DINO_CKPT và các đường dẫn Drive theo assets thật. Giữ cùng đường dẫn/config khi resume. Hai lock sẽ được nối tự động vào TINY/BOUNDARY sau khảo sát.

In [ ]:
DAY04 = PROJECT_DRIVE / 'msila_day04'
DATA = DAY04 / 'data/full_screen_v3'
CACHE = DATA / 'feature_cache'
TRAIN = DATA / 'records/train_core.json'
DEV = DATA / 'records/dev_synthetic.json'
MASKS = DATA / 'masks'
DINO_REPO = Path('/content/dinov3')
DINO_CKPT = PROJECT_DRIVE / 'weights/dinov3_vits16_pretrain_lvd1689m.pth'
SELECTION = DAY04 / 'e8_full_v3/selection_report.json'
PROTOCOL = DAY04 / 'configs/full_v3/day04_full_seed_42.yaml'
if not PROTOCOL.is_file():
    PROTOCOL = PROJECT_ROOT / 'configs/day05_day04_full_v3_protocol.yaml'
    print('Dùng bản protocol trích từ cells 7,28 notebook Day 04:', PROTOCOL)
OUT = PROJECT_DRIVE / 'outputs/day05/full_train'
# Sửa theo RAW_ROOT/CATEGORY_LOCAL_ROOTS thật; cần raw cho DEV replay.
RAW = Path('/content/msila_data/mvtec_ad_2')
DEV_EXPORT = PROJECT_DRIVE / 'msila_day05/native_dev'
INPUTS = DEV_EXPORT / 'dev_inputs.json'
TINY = OUT / 'protocols/tiny_protocol.lock.json'
BOUNDARY = OUT / 'protocols/boundary_protocol.lock.json'

COMMON = ['--cache-dir',str(CACHE),'--train-records',str(TRAIN),'--val-records',str(DEV),
    '--mask-root',str(MASKS),'--dinov3-repo',str(DINO_REPO),'--dino-checkpoint',str(DINO_CKPT),
    '--adapter-selection-report',str(SELECTION),'--training-protocol',str(PROTOCOL),
    '--output-root',str(OUT),'--category','fabric','--seed','42','--device','cuda:0','--resume']
def run(module, *args):
    cmd = [sys.executable,'-m',module,*map(str,args)]
    print(subprocess.list2cmdline(cmd))
    subprocess.run(cmd,cwd=PROJECT_ROOT,check=True)

## 3. Cấu hình khảo sát GT

Các mức khảo sát chỉ dùng GT. Có thể sửa trước khi chạy. Tolerance contour cần căn cứ annotation/protocol; coverage GT không tự chọn được tolerance. Khi resume với lock đã có, giữ các giá trị selected khớp lock và để EXPORT_LOCKED_PROTOCOLS=False.

In [ ]:
# 3. Cấu hình duy nhất cần sửa.
from pathlib import Path
from datetime import datetime, timezone
INPUT_MODE = 'manifest'  # Chỉ GT native cho workflow gộp.
MANIFEST_PATH = INPUTS
DAY04_RECORDS_PATH = DEV
DAY04_MASK_ROOT = MASKS
CATEGORIES = ['fabric']  # Có thể thêm 'vial', 'wallplugs'; không gộp split.

# Chỉ dùng khi INPUT_MODE='mask_directory'.
MASK_DIRECTORY = PROJECT_DRIVE / 'msila_day05/native_dev/native_masks/fabric'
DIRECTORY_CATEGORY = 'fabric'
DIRECTORY_SPLIT = 'dev_synthetic'
DIRECTORY_MASKS_ARE_NATIVE = False  # True chỉ khi đã xác nhận mask ở độ phân giải ảnh gốc.

TINY_THRESHOLDS_PX = [256, 512, 1024, 2048]
BAND_WIDTHS_PX = [8, 16, 32, 64]
SELECTED_TINY_AREA_PX = 1024
SELECTED_BAND_WIDTH_PX = 32
SELECTED_TOLERANCE_PX = 2  # Không suy ra độ chính xác contour từ đếm GT.
MAX_PREVIEW_IMAGES = 3
OUTPUT_ROOT = OUT / 'gt_protocol_review' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')

## 4. Audit cache và export DEV native

Replay kiểm tra mask crop và sáu DINO features với cache Day 04 trước khi xuất ảnh/GT native. Không cần train R0/R1/R2 để chạy bước này. Phải có raw sources và đầy đủ categories của records Day 04.

In [ ]:
run('scripts.run_day05_pipeline','--stage','audit','--dry-run',*COMMON)

In [ ]:
run('scripts.build_day05_cache','--export-dev-only','--data-root',RAW,
    '--categories','fabric,vial,wallplugs','--cache-dir',CACHE,'--train-records',TRAIN,
    '--val-records',DEV,'--mask-root',MASKS,'--dinov3-repo',DINO_REPO,
    '--dino-checkpoint',DINO_CKPT,'--output-root',DEV_EXPORT,'--device','cuda:0')

## 5. Khảo sát GT native: diện tích, coverage và preview

8-connectivity; tiny dùng area <= cutoff; boundary chọn component giao dải sát bốn mép ảnh. Không đọc maps của candidate.

In [ ]:
# 4. Helpers độc lập với repo; binary convention giống evaluator TV2.
import csv, hashlib, html, json, math
import numpy as np
from PIL import Image
from scipy.ndimage import label, find_objects
import matplotlib.pyplot as plt

STRUCTURE_8 = np.ones((3, 3), dtype=np.uint8)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def resolve_path(base, value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (Path(base) / path).resolve()

def read_gt(path):
    path = Path(path)
    if path.suffix.lower() == '.npy':
        array = np.load(path, allow_pickle=False)
    else:
        with Image.open(path) as image:
            array = np.array(image)
    if array.ndim != 2 or array.size == 0 or array.dtype.kind not in 'buif':
        raise ValueError(f'GT phải là mask 2-D numeric: {path}, shape={array.shape}')
    if not np.isfinite(array).all():
        raise ValueError(f'GT chứa NaN/Inf: {path}')
    values = set(np.unique(array).tolist())
    if not (values <= {0, 1} or values <= {0, 255}):
        raise ValueError(f'GT chỉ được chứa 0/1 hoặc 0/255: {path}, values={values}')
    return array != 0

def component_statistics(gt):
    labels, count = label(gt, structure=STRUCTURE_8)
    areas = np.bincount(labels.ravel(), minlength=count + 1)
    height, width = gt.shape
    regions = []
    for region_id, box in enumerate(find_objects(labels), start=1):
        if box is None:
            continue
        ys, xs = box
        # Min distance đến frame = min tọa độ cực trị của component.
        # Region giao band rộng w <=> min_border_distance_px < w (strict).
        distance = min(ys.start, xs.start, height - ys.stop, width - xs.stop)
        regions.append(dict(region_id=region_id, area_px=int(areas[region_id]),
                            area_fraction=float(areas[region_id] / gt.size),
                            min_border_distance_px=int(distance),
                            bbox_xyxy=[xs.start, ys.start, xs.stop, ys.stop]))
    return regions

def show_table(rows, limit=50):
    if not rows:
        print('(Không có dòng)')
        return
    fields = list(rows[0])
    output = '<table><thead><tr>' + ''.join('<th>' + html.escape(str(k)) + '</th>' for k in fields) + '</tr></thead><tbody>'
    for row in rows[:limit]:
        output += '<tr>' + ''.join('<td>' + html.escape(str(row.get(k, ''))) + '</td>' for k in fields) + '</tr>'
    output += '</tbody></table>'
    try:
        from IPython.display import display, HTML
    except ImportError:
        print(json.dumps(rows[:limit], ensure_ascii=False, indent=2))
    else:
        display(HTML(output))
    if len(rows) > limit:
        print(f'Hiển thị {limit}/{len(rows)} dòng; CSV chứa toàn bộ.')

def positive_integer(value, name, allow_zero=False):
    if isinstance(value, bool) or not isinstance(value, int) or value < (0 if allow_zero else 1):
        raise ValueError(f'{name} phải là số nguyên ' + ('không âm.' if allow_zero else 'dương.'))

In [ ]:
# 5. Chọn mẫu GT. Không load image hoặc prediction.
if not CATEGORIES or len(CATEGORIES) != len(set(CATEGORIES)):
    raise ValueError('CATEGORIES phải có tên category duy nhất.')
for value in TINY_THRESHOLDS_PX + [SELECTED_TINY_AREA_PX]:
    positive_integer(value, 'tiny_area_px')
for value in BAND_WIDTHS_PX + [SELECTED_BAND_WIDTH_PX]:
    positive_integer(value, 'band_width_px')
positive_integer(SELECTED_TOLERANCE_PX, 'tolerance_px', allow_zero=True)
TINY_THRESHOLDS_PX = sorted(set(TINY_THRESHOLDS_PX + [SELECTED_TINY_AREA_PX]))
BAND_WIDTHS_PX = sorted(set(BAND_WIDTHS_PX + [SELECTED_BAND_WIDTH_PX]))
mode = INPUT_MODE
if mode == 'auto':
    if MANIFEST_PATH.is_file():
        mode = 'manifest'
    elif DAY04_RECORDS_PATH.is_file():
        mode = 'day04_records'
    else:
        raise FileNotFoundError(f'Không tìm thấy native manifest: {MANIFEST_PATH}\nHoặc Day 04 records: {DAY04_RECORDS_PATH}\nSửa cell 3 theo path thật, hoặc dùng mask_directory.')
samples, input_identity = [], {}
if mode == 'manifest':
    payload = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    if payload.get('schema') not in {'msila.day05.inference_inputs.v1', 'msila.day05.metric_manifest.v1'}:
        raise ValueError('Cần dev_inputs.json hoặc metric manifest Day 05 có original_hw native.')
    SPLIT = payload['split']
    if SPLIT != 'dev_synthetic':
        raise ValueError('Notebook chọn protocol chỉ dùng dev_synthetic; không chọn ngưỡng bằng TEST.')
    COORDINATE_SCOPE = 'native_original_image'
    for sample in payload['samples']:
        if sample['category'] not in CATEGORIES:
            continue
        path = resolve_path(MANIFEST_PATH.parent, sample['gt_mask'])
        samples.append(dict(image_id=sample['image_id'], category=sample['category'], gt_path=str(path),
                            expected_hw=sample['original_hw'], expected_gt_sha256=sample.get('gt_mask_sha256')))
    input_identity = dict(path=str(MANIFEST_PATH.resolve()), sha256=sha256(MANIFEST_PATH), schema=payload['schema'])
elif mode == 'day04_records':
    payload = json.loads(DAY04_RECORDS_PATH.read_text(encoding='utf-8'))
    if not isinstance(payload, list):
        raise ValueError('Day 04 records phải là JSON list.')
    SPLIT, COORDINATE_SCOPE = 'dev_synthetic', 'day04_local_crop'
    for sample in payload:
        if sample['category'] not in CATEGORIES:
            continue
        record_split = sample.get('split', sample.get('meta', {}).get('split'))
        if record_split is not None and record_split != SPLIT:
            raise ValueError(f'Record khác DEV: {sample["image_id"]}')
        samples.append(dict(image_id=sample['image_id'], category=sample['category'],
                            gt_path=str(resolve_path(DAY04_MASK_ROOT, sample['mask_path'])),
                            expected_hw=sample.get('mask_hw'), expected_gt_sha256=None))
    input_identity = dict(path=str(DAY04_RECORDS_PATH.resolve()), sha256=sha256(DAY04_RECORDS_PATH),
                          mask_root=str(DAY04_MASK_ROOT.resolve()))
elif mode == 'mask_directory':
    if CATEGORIES != [DIRECTORY_CATEGORY]:
        raise ValueError('mask_directory yêu cầu CATEGORIES=[DIRECTORY_CATEGORY].')
    if DIRECTORY_SPLIT != 'dev_synthetic':
        raise ValueError('Không chọn protocol từ TEST; directory phải là GT dev_synthetic.')
    if not MASK_DIRECTORY.is_dir():
        raise FileNotFoundError(MASK_DIRECTORY)
    SPLIT = DIRECTORY_SPLIT
    COORDINATE_SCOPE = 'native_original_image' if DIRECTORY_MASKS_ARE_NATIVE else 'unverified_mask_coordinates'
    extensions = {'.png', '.bmp', '.tif', '.tiff', '.npy'}
    for path in sorted(MASK_DIRECTORY.rglob('*')):
        if path.is_file() and path.suffix.lower() in extensions:
            samples.append(dict(image_id=path.relative_to(MASK_DIRECTORY).as_posix(), category=DIRECTORY_CATEGORY,
                                gt_path=str(path.resolve()), expected_hw=None, expected_gt_sha256=None))
    input_identity = dict(path=str(MASK_DIRECTORY.resolve()), native_confirmed=DIRECTORY_MASKS_ARE_NATIVE,
                          note='Mask files only; normal images without mask files are not counted.')
else:
    raise ValueError(f'INPUT_MODE không hợp lệ: {mode}')
if not samples:
    raise ValueError(f'Không có GT cho CATEGORIES={CATEGORIES}, input={mode}')
ids = [sample['image_id'] for sample in samples]
if len(ids) != len(set(ids)):
    raise ValueError('image_id trùng; kiểm tra nguồn dữ liệu.')
if {sample['category'] for sample in samples} != set(CATEGORIES):
    raise ValueError('Có category yêu cầu nhưng không có GT; kiểm tra CATEGORIES.')
print(f'Input={mode}; split={SPLIT}; coordinates={COORDINATE_SCOPE}; samples={len(samples)}')
if COORDINATE_SCOPE != 'native_original_image':
    print('KHẢO SÁT CROP/CHƯA XÁC NHẬN NATIVE: không xuất protocol native Day 05.')

In [ ]:
# 6. Đọc mỗi mask một lần. Không resize, không bỏ ảnh normal có mask rỗng.
image_rows, region_rows = [], []
for index, sample in enumerate(samples, start=1):
    gt = read_gt(sample['gt_path'])
    if sample['expected_hw'] is not None and list(gt.shape) != list(sample['expected_hw']):
        raise ValueError(f'GT shape mismatch: {sample["image_id"]}, {gt.shape} != {sample["expected_hw"]}')
    digest = sha256(sample['gt_path'])
    if sample['expected_gt_sha256'] is not None and digest != sample['expected_gt_sha256']:
        raise ValueError(f'GT checksum mismatch: {sample["image_id"]}')
    regions = component_statistics(gt)
    shared = dict(image_id=sample['image_id'], category=sample['category'])
    image_rows.append(dict(**shared, height=gt.shape[0], width=gt.shape[1], foreground_px=int(gt.sum()),
                           n_regions=len(regions), gt_path=sample['gt_path'], gt_sha256=digest,
                           split=SPLIT, coordinate_scope=COORDINATE_SCOPE))
    region_rows.extend(dict(**shared, **region) for region in regions)
    if index % 25 == 0 or index == len(samples):
        print(f'{index}/{len(samples)} masks; {len(region_rows)} vùng lỗi')
print('Pixel chạm góc được tính cùng vùng. Ảnh normal: foreground_px=0, n_regions=0.')

In [ ]:
# 7. Tóm tắt dữ liệu và phân vị diện tích GT (không tự chọn percentile làm ngưỡng).
summary_rows = []
for category in CATEGORIES:
    images = [row for row in image_rows if row['category'] == category]
    regions = [row for row in region_rows if row['category'] == category]
    areas = np.array([row['area_px'] for row in regions], dtype=np.int64)
    sizes = sorted({(row['height'], row['width']) for row in images})
    row = dict(category=category, coordinate_scope=COORDINATE_SCOPE, n_mask_files=len(images),
               n_zero_masks=sum(row['foreground_px'] == 0 for row in images),
               n_anomalous_masks=sum(row['foreground_px'] > 0 for row in images),
               n_regions=len(regions), resolutions='; '.join(f'{h}x{w}' for h, w in sizes))
    for percentile in (0, 10, 25, 50, 75, 90, 100):
        row[f'area_P{percentile}_px'] = round(float(np.percentile(areas, percentile)), 2) if areas.size else None
    summary_rows.append(row)
show_table(summary_rows)
print('Phân vị là thống kê từng component, không phải diện tích tổng mỗi ảnh.')

In [ ]:
# 8. Coverage tiny theo các mốc: area_px <= threshold.
tiny_rows = []
for category in CATEGORIES:
    regions = [row for row in region_rows if row['category'] == category]
    for threshold in TINY_THRESHOLDS_PX:
        selected = [row for row in regions if row['area_px'] <= threshold]
        tiny_rows.append(dict(category=category, tiny_area_px=threshold, n_regions=len(regions),
                              n_tiny_regions=len(selected), n_non_tiny_regions=len(regions)-len(selected),
                              tiny_region_percent=round(100 * len(selected) / len(regions), 2) if regions else None,
                              n_images_with_tiny=len({row['image_id'] for row in selected}),
                              status='OK' if selected else 'NO_GT_TINY_REGIONS'))
show_table(tiny_rows)

In [ ]:
# 9. Coverage boundary theo dải sát BỐN MÉP ẢNH của tọa độ đang đọc.
# min_border_distance_px < band_width_px; toàn bộ component được chọn khi giao band.
boundary_rows = []
for category in CATEGORIES:
    regions = [row for row in region_rows if row['category'] == category]
    images = [row for row in image_rows if row['category'] == category]
    for width in BAND_WIDTHS_PX:
        selected = [row for row in regions if row['min_border_distance_px'] < width]
        zone_fractions = [1 - max(row['height']-2*width, 0)*max(row['width']-2*width, 0)/(row['height']*row['width']) for row in images]
        boundary_rows.append(dict(category=category, band_width_px=width, n_regions=len(regions),
                                  n_boundary_regions=len(selected), n_non_boundary_regions=len(regions)-len(selected),
                                  boundary_region_percent=round(100*len(selected)/len(regions), 2) if regions else None,
                                  n_images_with_boundary=len({row['image_id'] for row in selected}),
                                  mean_zone_image_percent=round(100*float(np.mean(zone_fractions)), 2),
                                  n_images_zone_covers_all=sum(2*width >= min(row['height'], row['width']) for row in images),
                                  status='OK' if selected else 'NO_GT_BOUNDARY_REGIONS'))
show_table(boundary_rows)
print('Band quá lớn có thể phủ toàn ảnh; xem n_images_zone_covers_all và preview.')

In [ ]:
# 10. Biểu đồ GT và coverage. Không có vùng lỗi thì vẫn xuất báo cáo bình thường.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for category in CATEGORIES:
    areas = np.array([row['area_px'] for row in region_rows if row['category'] == category])
    if areas.size:
        axes[0].hist(np.log10(areas), bins=20, alpha=0.5, label=category)
    tr = [row for row in tiny_rows if row['category'] == category]
    br = [row for row in boundary_rows if row['category'] == category]
    axes[1].plot([row['tiny_area_px'] for row in tr], [row['n_tiny_regions'] for row in tr], 'o-', label=category)
    axes[2].plot([row['band_width_px'] for row in br], [row['n_boundary_regions'] for row in br], 'o-', label=category)
axes[0].axvline(math.log10(SELECTED_TINY_AREA_PX), color='black', linestyle='--', label='Selected threshold')
axes[0].set(xlabel='log10(component area in pixels)', ylabel='Region count', title='GT area distribution')
axes[1].set(xlabel='Tiny area threshold (pixels²)', ylabel='Tiny region count', title='Tiny coverage')
axes[2].set(xlabel='Image-border band (pixels)', ylabel='Boundary region count', title='Image-edge coverage')
for axis in axes:
    axis.grid(alpha=0.2)
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend()
fig.suptitle(f'{SPLIT} | {COORDINATE_SCOPE} | GT only')
fig.tight_layout()
coverage_figure = fig
plt.show()

In [ ]:
# 11. Preview: ưu tiên ảnh lỗi gần mép, rồi vùng nhỏ; không random/cherry-pick prediction.
anomalous_ids = {row['image_id'] for row in region_rows}
ordered = sorted(region_rows, key=lambda row: (row['min_border_distance_px'], row['area_px'], row['image_id']))
preview_ids = list(dict.fromkeys(row['image_id'] for row in ordered))[:MAX_PREVIEW_IMAGES]
preview_figures = []
for image_id in preview_ids:
    sample = next(sample for sample in samples if sample['image_id'] == image_id)
    gt = read_gt(sample['gt_path'])
    labels, count = label(gt, structure=STRUCTURE_8)
    selected = [row for row in region_rows if row['image_id'] == image_id]
    tiny_ids = [row['region_id'] for row in selected if row['area_px'] <= SELECTED_TINY_AREA_PX]
    boundary_ids = [row['region_id'] for row in selected if row['min_border_distance_px'] < SELECTED_BAND_WIDTH_PX]
    ys, xs = np.ogrid[:gt.shape[0], :gt.shape[1]]
    band = (ys < SELECTED_BAND_WIDTH_PX) | (ys >= gt.shape[0]-SELECTED_BAND_WIDTH_PX) | (xs < SELECTED_BAND_WIDTH_PX) | (xs >= gt.shape[1]-SELECTED_BAND_WIDTH_PX)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis in axes:
        axis.imshow(gt, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        axis.set_axis_off()
    axes[0].set_title(f'GT: {count} regions')
    axes[1].imshow(np.ma.masked_where(~np.isin(labels, tiny_ids), np.ones(gt.shape)), cmap='autumn', vmin=0, vmax=1, alpha=0.8, interpolation='nearest')
    axes[1].set_title(f'Tiny <= {SELECTED_TINY_AREA_PX}px²: {len(tiny_ids)}')
    axes[2].imshow(np.ma.masked_where(~band, np.ones(gt.shape)), cmap='Blues', vmin=0, vmax=1, alpha=0.3, interpolation='nearest')
    axes[2].imshow(np.ma.masked_where(~np.isin(labels, boundary_ids), np.ones(gt.shape)), cmap='autumn', vmin=0, vmax=1, alpha=0.8, interpolation='nearest')
    axes[2].set_title(f'Border band {SELECTED_BAND_WIDTH_PX}px: {len(boundary_ids)}')
    fig.suptitle(f'{image_id} | {COORDINATE_SCOPE}')
    fig.tight_layout()
    preview_figures.append((image_id, fig))
    plt.show()
if not preview_ids:
    print('Tất cả mask rỗng: không có defect để preview.')

**Đọc kết quả:**
- `n_tiny_regions` và `n_boundary_regions`: số vùng GT đủ điều kiện; `n_images_with_*`: số ảnh chứa nhóm đó.
- `NO_GT_*_REGIONS`: không có vùng để tính metric của nhóm; không phải training fail. Không tăng ngưỡng chỉ để tạo một kết quả có lợi cho candidate.
- Xem số vùng theo từng category, tỷ lệ band phủ ảnh và preview. Không có số lượng vùng tối thiểu phổ quát bảo đảm một kết luận đáng tin cậy; nhóm rất ít vùng cần báo rõ số lượng.
- `tolerance_px` không thể chọn chỉ bằng histogram diện tích. Cần căn cứ độ chính xác annotation hoặc protocol contour đã định trước; notebook giữ nó là tham số do bạn nhập.
- Crop Day 04 không trả lời coverage sát mép **ảnh gốc**. Muốn native DEV, chạy bước **DEV replay/export** trong runbook Day 05, rồi chạy lại notebook ở mode `manifest`. Export không cần đợi train R0/R1/R2 hoàn tất, nhưng workflow replay hiện tại cần raw sources, cache và DINO để kiểm tra tính nhất quán.

In [ ]:
# 12. Xuất CSV, biểu đồ và provenance; không ghi đè protocol hiện tại của project.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
def write_csv(path, rows, fields=None):
    if fields is None:
        fields = list(rows[0]) if rows else []
    with Path(path).open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

write_csv(OUTPUT_ROOT/'gt_images.csv', image_rows)
write_csv(OUTPUT_ROOT/'gt_regions.csv', region_rows,
          ['image_id','category','region_id','area_px','area_fraction','min_border_distance_px','bbox_xyxy'])
write_csv(OUTPUT_ROOT/'gt_summary.csv', summary_rows)
write_csv(OUTPUT_ROOT/'tiny_coverage.csv', tiny_rows)
write_csv(OUTPUT_ROOT/'boundary_coverage.csv', boundary_rows)
coverage_figure.savefig(OUTPUT_ROOT/'gt_coverage.png', dpi=150, bbox_inches='tight')
for index, (image_id, figure) in enumerate(preview_figures, start=1):
    figure.savefig(OUTPUT_ROOT/f'gt_preview_{index:02d}.png', dpi=150, bbox_inches='tight')
report = dict(schema='msila.day05.gt_protocol_review.v1', created_utc=datetime.now(timezone.utc).isoformat(),
              input_mode=mode, input_identity=input_identity, split=SPLIT, coordinate_scope=COORDINATE_SCOPE,
              categories=CATEGORIES, n_masks=len(image_rows), n_regions=len(region_rows), connectivity=8,
              prediction_used=False, tiny_thresholds_px=TINY_THRESHOLDS_PX, band_widths_px=BAND_WIDTHS_PX,
              selected_proposal=dict(tiny_area_px=SELECTED_TINY_AREA_PX, band_width_px=SELECTED_BAND_WIDTH_PX,
                                     tolerance_px=SELECTED_TOLERANCE_PX),
              samples=[dict(image_id=row['image_id'], category=row['category'], gt_path=row['gt_path'],
                            gt_sha256=row['gt_sha256'], mask_hw=[row['height'],row['width']]) for row in image_rows])
report_path = OUTPUT_ROOT/'gt_review.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
print('Đã xuất:', OUTPUT_ROOT)
print('GT review SHA256:', sha256(report_path))

In [ ]:
# 13. Xuất PROPOSAL native. Vẫn false, không tự khóa scientific protocol.
# Với crop/unverified coordinates, chỉ có CSV/plots; không xuất JSON protocol native.
if COORDINATE_SCOPE == 'native_original_image':
    basis = f'GT-only coverage review {report_path}; SHA256={sha256(report_path)}; split={SPLIT}; categories={CATEGORIES}.'
    tiny_proposal = dict(tiny_area_px=SELECTED_TINY_AREA_PX, area_unit='original_image_pixels', connectivity=8,
                         max_fpr=0.05, locked_before_candidate_results=False,
                         threshold_basis=f'PROPOSAL: fixed native component area cutoff. {basis}')
    boundary_proposal = dict(boundary_mode='image_border_band', band_width_px=SELECTED_BAND_WIDTH_PX,
                             connectivity=8, max_fpr=0.05, prediction_threshold=0.5,
                             tolerance_px=SELECTED_TOLERANCE_PX, locked_before_candidate_results=False,
                             rule_basis=f'PROPOSAL: fixed band at four image edges; contour tolerance is a user proposal, not inferred from coverage. {basis}')
    for name, payload in [('tiny_protocol.proposal.json', tiny_proposal), ('boundary_protocol.proposal.json', boundary_proposal)]:
        path = OUTPUT_ROOT/name
        path.write_text(json.dumps(payload, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
        print(path)
else:
    print('Không xuất protocol native từ crop/chưa xác nhận tọa độ. Dùng báo cáo để khảo sát, rồi chuyển sang native manifest.')

## 6. Khóa protocol

Xem GT ở trên và thống nhất căn cứ area cutoff, band width, contour tolerance trước khi xem candidate results. Điền hai basis và bật EXPORT_LOCKED_PROTOCOLS khi điều đó thực sự đúng. Nếu đã có lock đúng dưới OUT/protocols để resume, giữ False và chạy cell nối protocol tiếp theo. Lock đã có nội dung khác sẽ được giữ nguyên; dùng output experiment mới khi đổi protocol.

In [ ]:
# 14. Tùy chọn: chỉ đổi True khi THỰC SỰ đã thống nhất trước candidate results.
EXPORT_LOCKED_PROTOCOLS = False
TINY_LOCK_BASIS = ''  # Ví dụ: mốc area nghiên cứu + coverage native đã khảo sát, không dựa trên score.
BOUNDARY_LOCK_BASIS = ''  # Vì sao dùng image-edge band này và tolerance này (annotation/protocol).
if EXPORT_LOCKED_PROTOCOLS:
    if COORDINATE_SCOPE != 'native_original_image':
        raise ValueError('Chỉ khóa protocol native từ GT native.')
    if not TINY_LOCK_BASIS.strip() or not BOUNDARY_LOCK_BASIS.strip():
        raise ValueError('Điền căn cứ thật cho cả tiny và boundary trước khi khóa.')
    evidence = f' GT-only report SHA256={sha256(report_path)}.'
    tiny_locked = dict(tiny_proposal, locked_before_candidate_results=True, threshold_basis=TINY_LOCK_BASIS.strip()+evidence)
    boundary_locked = dict(boundary_proposal, locked_before_candidate_results=True, rule_basis=BOUNDARY_LOCK_BASIS.strip()+evidence)
    for name, payload in [('tiny_protocol.lock.json', tiny_locked), ('boundary_protocol.lock.json', boundary_locked)]:
        path = OUTPUT_ROOT/name
        encoded = json.dumps(payload, ensure_ascii=False, indent=2)+'\n'
        if path.exists() and path.read_text(encoding='utf-8') != encoded:
            raise FileExistsError(f'Không ghi đè lock khác nội dung: {path}; dùng thư mục báo cáo mới.')
        if not path.exists():
            with path.open('x', encoding='utf-8') as stream:
                stream.write(encoded)
        print('LOCK:', path, 'SHA256:', sha256(path))
else:
    print('Chưa khóa. CSV/plots và native proposals (nếu có) đã được xuất.')

## 7. Nối lock vào pipeline và kiểm tra

Cell này dùng lock vừa tạo, hoặc tái sử dụng lock đã có dưới OUT/protocols. Chưa khóa thì dừng ở đây; quay lại phần 6 sau khi thống nhất. Kiểm tra không thay cho việc đọc GT và khai báo trung thực thời điểm khóa.

In [ ]:
# Nối hai lock; không cần copy thủ công hoặc sửa TINY/BOUNDARY mỗi lần.
PROTOCOL_ROOT = OUT / 'protocols'
protocol_targets = [PROTOCOL_ROOT/'tiny_protocol.lock.json', PROTOCOL_ROOT/'boundary_protocol.lock.json']
protocol_sources = [OUTPUT_ROOT/'tiny_protocol.lock.json', OUTPUT_ROOT/'boundary_protocol.lock.json']
protocol_payloads = []
for source, target in zip(protocol_sources, protocol_targets):
    chosen = source if source.is_file() else target
    if not chosen.is_file():
        raise RuntimeError('Chưa có protocol lock. Xem coverage GT, điền basis và chạy phần 6; sau đó chạy lại cell này.')
    payload = json.loads(chosen.read_text(encoding='utf-8'))
    if payload.get('locked_before_candidate_results') is not True:
        raise ValueError(f'Protocol chưa khóa: {chosen}')
    protocol_payloads.append(payload)
tiny_lock, boundary_lock = protocol_payloads
if set(tiny_lock) != {'tiny_area_px','area_unit','connectivity','max_fpr','locked_before_candidate_results','threshold_basis'}:
    raise ValueError('Tiny protocol schema không khớp evaluator.')
if set(boundary_lock) != {'boundary_mode','band_width_px','connectivity','max_fpr','prediction_threshold','tolerance_px','locked_before_candidate_results','rule_basis'}:
    raise ValueError('Boundary protocol schema không khớp evaluator.')
if tiny_lock['area_unit'] != 'original_image_pixels' or boundary_lock['boundary_mode'] != 'image_border_band':
    raise ValueError('Workflow gộp dùng native area và image_border_band.')
for payload, basis in [(tiny_lock,'threshold_basis'), (boundary_lock,'rule_basis')]:
    if payload['connectivity'] != 8 or payload['max_fpr'] != 0.05:
        raise ValueError('Giữ connectivity=8 và max_fpr=0.05.')
    if not isinstance(payload[basis],str) or not payload[basis].strip():
        raise ValueError('Protocol thiếu căn cứ thật.')
if boundary_lock['prediction_threshold'] != 0.5:
    raise ValueError('Giữ prediction_threshold=0.5 theo Day 04.')
positive_integer(tiny_lock['tiny_area_px'], 'tiny_area_px')
positive_integer(boundary_lock['band_width_px'], 'band_width_px')
positive_integer(boundary_lock['tolerance_px'], 'tolerance_px', allow_zero=True)
if (tiny_lock['tiny_area_px'],boundary_lock['band_width_px'],boundary_lock['tolerance_px']) != (SELECTED_TINY_AREA_PX,SELECTED_BAND_WIDTH_PX,SELECTED_TOLERANCE_PX):
    raise ValueError('Ngưỡng selected trong khảo sát khác lock. Khi resume, đặt lại selected theo lock hiện có; khi đổi protocol, dùng experiment mới.')
if COORDINATE_SCOPE != 'native_original_image' or SPLIT != 'dev_synthetic' or CATEGORIES != ['fabric']:
    raise ValueError('Phase 1 yêu cầu GT native DEV và CATEGORIES=[fabric].')
# Kiểm tra cả hai file trước khi ghi để tránh ghi một lock rồi mới phát hiện conflict.
encoded_payloads = [json.dumps(payload,ensure_ascii=False,indent=2)+'\n' for payload in protocol_payloads]
for target, payload in zip(protocol_targets,protocol_payloads):
    if target.exists() and json.loads(target.read_text(encoding='utf-8')) != payload:
        raise FileExistsError(f'Giữ lock đã có, không ghi đè protocol khác: {target}. Dùng OUT mới cho experiment mới.')
PROTOCOL_ROOT.mkdir(parents=True,exist_ok=True)
for target, encoded in zip(protocol_targets,encoded_payloads):
    if not target.exists():
        with target.open('x',encoding='utf-8') as stream:
            stream.write(encoded)
TINY, BOUNDARY = protocol_targets
print('Tiny lock:',TINY,'SHA256:',sha256(TINY))
print('Boundary lock:',BOUNDARY,'SHA256:',sha256(BOUNDARY))
print('Protocol đã nối vào training/inference/evaluation workflow.')

## 8. Preflight và full training R0 → R1 → R2

Mỗi candidate chạy đủ 150 epochs với vits16. --resume bật sẵn; khi runtime ngắt, khôi phục assets/setup rồi tái sử dụng lock và chạy lại training. COMPLETE có checksum được skip; giữ code/config/paths.

In [ ]:
run('scripts.run_day05_pipeline','--stage','preflight',*COMMON)

In [ ]:
run('scripts.run_day05_pipeline','--stage','train',*COMMON)

## 9. Inference native DEV

Strict-load best.pt, Context extraction riêng cho R2, tiling512/overlap128/Hann stitch; xuất probability maps float32 native.

In [ ]:
run('scripts.run_day05_pipeline','--stage','inference',*COMMON,'--input-manifest',INPUTS,'--seg-f1-threshold','0.5')

## 10. Evaluation và kết quả

AU-PRO0.05 → tiny → boundary → region stats → comparison → efficiency → aggregation → representation lock. Hai lock được truyền tự động. TEST không dùng chọn representation.

In [ ]:
run('scripts.run_day05_pipeline','--stage','evaluate',*COMMON,'--input-manifest',INPUTS,
    '--seg-f1-threshold','0.5','--tiny-protocol',TINY,'--boundary-protocol',BOUNDARY)

In [ ]:
for candidate in ('R0','R1','R2'):
    p=OUT/'seed_42/fabric'/candidate/'run_manifest.json'
    if p.is_file():
        r=json.loads(p.read_text());print(candidate,r['status'],r.get('completed_epochs'),r.get('global_step'))
    else: print(candidate,'BLOCKED / chưa có run')
print('Native maps:', OUT/'maps/dev_synthetic')
print('Evaluation:', OUT/'evaluation/dev_synthetic')
print('Lock:', OUT/'evaluation/dev_synthetic/summary/representation_lock.yaml')

Tham khảo gốc: [COCO area ranges](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py) dùng small đến 32² pixel²; [DAVIS boundary F](https://github.com/davisvideochallenge/davis2017-evaluation/blob/master/davis2017/metrics.py) dùng tolerance mặc định 0.008×đường chéo; [Detectron2 Boundary IoU](https://github.com/facebookresearch/detectron2/blob/main/detectron2/evaluation/sem_seg_evaluation.py) dùng band contour 0.02×đường chéo. Band **contour** của Detectron2 không phải band **mép ảnh** trong notebook này. Các mốc khảo sát không được tự coi là protocol MVTec chuẩn.